In [6]:
import pandas as pd
import glob

# Get a list of all CSV files in the directory
csv_files = glob.glob('training_data/flipkart//*.csv')

# Create a list to hold the dataframes
dfs = []

# Read each CSV file and append it to the list of dataframes
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dfs.append(df)

# Concatenate all dataframes in the list
combined_df = pd.concat(dfs, ignore_index=True)

# Print the combined dataframe
# Extract a specific column
specific_column = combined_df['name'].dropna()

# Drop rows with non-existing values in the specific column
# todo wirte to a csv file
shuffled_df = specific_column.sample(frac=1).reset_index(drop=True)


In [96]:
import pandas as pd

df = pd.concat([pd.read_csv('training_data/products.txt', sep= '\t', names=["text","label"]), \
			   pd.read_csv('training_data/negatifs.txt', sep= '\t', names=["text","label"]), \
			   pd.read_csv('training_data/products_factorybuy.txt', sep= '\t', names=["text","label"]), \
			   pd.read_csv('training_data/negatifs_factorybuy.txt', sep= '\t', names=["text","label"])])


In [97]:
df.head()

,text,label
0,2.5-seat mod sofa w chaise,True
1,Modular loveseat,True
2,Sectional 6-seat crn/chaise,True
3,"Sectional, 4-seat corner",True
4,3-seat sofa,True


In [64]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [98]:
def process_data(row):
    text = row['text']
    text = str(text)
    text = ' '.join(text.split())

    encodings = tokenizer(text,truncation=True)

    label = 0
    if row['label'] == 1:
        label += 1

    encodings['label'] = label
    encodings['text'] = text

    return encodings

In [99]:
print(process_data(df.iloc[0]))

{'input_ids': [101, 1016, 1012, 1019, 1011, 2835, 16913, 10682, 1059, 15775, 5562, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'label': 1, 'text': '2.5-seat mod sofa w chaise'}


In [100]:
df.head()

,text,label
0,2.5-seat mod sofa w chaise,True
1,Modular loveseat,True
2,Sectional 6-seat crn/chaise,True
3,"Sectional, 4-seat corner",True
4,3-seat sofa,True


In [101]:
processed_data = []

for i in range(len(df)):
    processed_data.append(process_data(df.iloc[i]))

In [102]:
count = 0
for data in processed_data:
	if data['label'] == 1:
		count += 1
print(count)

7459


In [70]:
df.count()

text     19736
label    19736
dtype: int64

In [71]:
df["label"]

0        True
1        True
2        True
3        True
4        True
        ...  
5841    False
5842    False
5843    False
5844    False
5845    False
Name: label, Length: 19736, dtype: bool

In [103]:
from sklearn.model_selection import train_test_split

new_df = pd.DataFrame(processed_data)

train_df, valid_df = train_test_split(
    new_df,
    test_size=0.19,
    random_state=2022
)

In [104]:
import pyarrow as pa
from datasets import Dataset

train_hg = Dataset(pa.Table.from_pandas(train_df)).shuffle(seed=2022)
valid_hg = Dataset(pa.Table.from_pandas(valid_df)).shuffle(seed=2022)

In [105]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

loading configuration file config.json from cache at /home/oprisorandrei/.cache/huggingface/hub/models--bert-base-uncased/snapshots/1dbc166cf8765166998eff31ade2eb64c8a40076/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /home/oprisorandrei/.cache/huggingface/hub/models--bert-base-uncased/snapshots/1dbc166cf8765166998eff

In [107]:

import torch
from transformers import TrainingArguments, Trainer
import torch
from transformers import TrainingArguments, Trainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=25,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.02,
    logging_dir='./logs',  # Specify GPU device
)

trainer = Trainer(
    model=model,
    args=training_args,
	train_dataset=train_hg,
    eval_dataset=valid_hg,
    tokenizer=tokenizer
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [108]:
trainer.train()

The following columns in the training set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: __index_level_0__, text. If __index_level_0__, text are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
/home/oprisorandrei/.local/lib/python3.10/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 15986
  Num Epochs = 25
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 25000
  Number of trainable parameters = 109483778


  0%|          | 0/25000 [00:00<?, ?it/s]

: 

In [79]:
from torch.utils.data import DataLoader

model = AutoModelForSequenceClassification.from_pretrained('./model')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

data_loader = DataLoader(valid_hg, batch_size=16, shuffle=True)



The following columns in the evaluation set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: __index_level_0__, text. If __index_level_0__, text are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 3750
  Batch size = 64
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  0%|          | 0/59 [00:00<?, ?it/s]

{'eval_loss': 0.6953178644180298,
 'eval_runtime': 78.0917,
 'eval_samples_per_second': 48.02,
 'eval_steps_per_second': 0.756}

In [80]:
from sklearn.metrics import classification_report

model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

In [94]:
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

new_model = AutoModelForSequenceClassification.from_pretrained('./results/checkpoint-4000/').to(device)

loading configuration file ./results/checkpoint-4000/config.json
Model config BertConfig {
  "_name_or_path": "./results/checkpoint-4000/",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "torch_dtype": "float32",
  "transformers_version": "4.26.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file ./results/checkpoint-4000/pytorch_model.bin
All model checkpoint weights were used when initializing BertForSequenceClassification.

All the weights of B

In [95]:
new_model.save_pretrained('./model4epochs/')

Configuration saved in ./model4epochs/config.json
Model weights saved in ./model4epochs/pytorch_model.bin


In [69]:
# Load the BERT model weights
plm = torch.load('./results/checkpoint-5000/pytorch_model.bin')

# Accessing a specific tensor
word_embeddings_weight = plm['classifier.weight']
len(word_embeddings_weight[1])

768

In [16]:

from transformers import AutoTokenizer

new_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [84]:
import torch
import numpy as np


def get_prediction(tokenizer, model, text):
	device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
	encoding = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=16)
	encoding = {k: v.to(device) for k, v in encoding.items()}

	outputs = model(**encoding)

	logits = outputs.logits
	probs = torch.nn.functional.softmax(logits, dim=-1)
	sigmoid = torch.nn.Sigmoid()
	probs = sigmoid(logits.squeeze().detach())
	probs = probs.detach().numpy()
	if np.argmax(probs, axis=-1) == 1:
		return True
	else:
		return False
		

In [85]:
def evaluate_model(model, dataset, tokenizer):
	texts = dataset['text']
	true_labels = dataset['label']

	predictions = [get_prediction(tokenizer, model, text) for text in texts]

	report = classification_report(true_labels, predictions)
	return report
	

In [87]:
report = evaluate_model(new_model, valid_df, new_tokenizer)

In [88]:
print (report)

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      2291
           1       0.99      1.00      0.99      1459

    accuracy                           1.00      3750
   macro avg       1.00      1.00      1.00      3750
weighted avg       1.00      1.00      1.00      3750



In [60]:
import time
start = time.time()
print(get_prediction(new_tokenizer, new_model, "Telephone 65x20 cm"))
print (time.time() - start)

[0.00911242 0.99412596]
0.04905390739440918


In [49]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup as bs
import torch
import numpy as np
import threading
from urllib.parse import urlparse
from transformers import AutoTokenizer, AutoModelForSequenceClassification

options = Options()
service = Service(ChromeDriverManager().install())



class crawler:
	def __init__(self):
		self.lock = None
		self.tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
		device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.model = AutoModelForSequenceClassification.from_pretrained('./model').to(device)

	def get_prediction(self, tokenizer, model, text):
		device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		encoding = tokenizer(text, return_tensors='pt', truncation=True)
		encoding = {k: v.to(device) for k, v in encoding.items()}

		outputs = model(**encoding)

		logits = outputs.logits
		probs = torch.nn.functional.softmax(logits, dim=-1)
		sigmoid = torch.nn.Sigmoid()
		probs = sigmoid(logits.squeeze().detach())
		probs = probs.detach().numpy()
		if np.argmax(probs, axis=-1) == 1:
			return text

	def scraper(self, urls):
		products = []
		driver = webdriver.Chrome(service=service, options=options)
		driver.set_page_load_timeout
		for url in urls:
			print(f"Crawling {url}...")
			products = []
			try:
				driver.get(url)
				# Get the page source
				products = []
				page_source = driver.page_source
				soup = bs(page_source, 'html.parser')
				links = soup.find_all('a')
				headers = soup.find_all(['h1','h2','h3','h4','h5','h6', 'p'])

				for header in headers:
					if self.get_prediction(self.tokenizer, self.model, header.get_text(strip=True)) is not None:
						products.append(header.get_text(strip=True))

				for link in links:
					text = link.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
					if text is None:
						texts = link.get_text(separator=' | ',strip=True).split(' | ')
						for text in texts:
							if self.get_prediction(self.tokenizer, self.model, text) is not None:
								products.append(text)
								break
			except Exception as e:
				print(f"Exception occurred while crawling: {str(e)}")
				continue
			
			if products and self.lock is not None:
				with self.lock:
					with open('products.txt', 'a') as f:
						f.write(urlparse(url).netloc + '\n')
						for product in products:
							f.write(product + '\n')
			elif products:
				with open('products.txt', 'a') as f:
					f.write(urlparse(url).netloc + '\n')
					for product in products:
						f.write(product + '\n')
		driver.quit()



	def crawl(self, urls, threaded, num_workers = None):
		if threaded:
			threads = []
			self.lock = threading.Lock()
			for _ in range(num_workers):
				t = threading.Thread(target=self.scraper, args=(urls,))
				t.start()
				threads.append(t)
			
			for thread in threads:
				thread.join()
		else:
			self.scraper(urls)
	 